### KOMAP Sample Code
*Adapted from README in KOMAP GitHub repository*

In [7]:
library(KOMAP)
library(dplyr)
library(mclust)

Package 'mclust' version 6.1.2
Type 'citation("mclust")' for citing this R package in publications.


Attaching package: ‘mclust’


The following object is masked from ‘package:dplyr’:

    count




In [ ]:
head(fake_ehr)

,patient_num,days_since_admission,concept_type,concept_code
,<int>,<int>,<chr>,<chr>
1,10,1,DIAG-ICD9,285
2,10,1,DIAG-ICD9,599
3,10,1,LAB-LOINC,6690-2
4,10,1,LAB-LOINC,6690-2
5,10,1,LAB-LOINC,2532-0
6,10,1,LAB-LOINC,2160-0


In [3]:
data(ehr_data)
data(rollup_dict)
data(filter_df)
input_cov <- gen_cov_input(ehr_data, rollup_dict, filter_df, main_surrogates = 'PheCode:250', train_ratio = 1/2)

Joining with `by = join_by(code)`
Joining with `by = join_by(code)`


In [ ]:
input_cov$train_cov[1:3,1:3]

,corrupt_PheCode:250,LAB-LOINC:1742-6,LAB-LOINC:1920-8
corrupt_PheCode:250,0,0,0.00000000
LAB-LOINC:1742-6,0,0,0.00000000
LAB-LOINC:1920-8,0,0,0.02489652


In [5]:
input_cov$valid_cov[1:3,1:3]

,corrupt_PheCode:250,LAB-LOINC:1742-6,LAB-LOINC:1920-8
corrupt_PheCode:250,0.1351274,0.3256309,0.10539262
LAB-LOINC:1742-6,0.3256309,0.7847075,0.25397584
LAB-LOINC:1920-8,0.1053926,0.2539758,0.08220098


In [11]:
codify.feature <- codify_RA$Variable[codify_RA$high_confidence_level == 1]
nlp.feature <- cui_RA$cui[cui_RA$high_confidence_level == 1]
input.cov.train <- cov_RA_train_long
input.cov.valid <- cov_RA_valid_long

target.code <- 'PheCode:714.1'
target.cui <- 'C0003873'
nm.corrupt.code <- 'corrupt_mainICD'
nm.corrupt.cui <- 'corrupt_mainNLP'
nm.utl <- 'utl'
nm.pi <- 'pi'
nm.id <- 'patient_num'
nm.y <- 'Y'
dat.part <- dat_part

## When the input is in a long format:
out_input_long <- KOMAP_corrupt(input.cov.train, input.cov.valid, is.wide = FALSE, target.code, target.cui, 
                                nm.disease = 'RA', nm.utl, nm.multi = NULL, nm.corrupt.code = nm.corrupt.code, 
                                nm.corrupt.cui = nm.corrupt.cui, dict_RA, pred = FALSE, 
                                eval.real = FALSE, eval.sim = FALSE)


Input long format data, transformed to wide format covariance matrix (45 unique nodes).

Check feature format in `input.cov`...

Num of total feat: 45


Finish estimating coefficients.



In [12]:
head(out_input_long$est$long_df)

disease,method,target,feat,desc,coeff
<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>
PheCode:714.1,mainICD + codify,PheCode:714.1,PheCode:714.1,rheumatoid arthritis,0.62534523
PheCode:714.1,mainICD + codify,PheCode:714.1,utl,Healthcare Utility,-0.20183315
PheCode:714.1,mainICD + codify,PheCode:714.1,C0003873,Rheumatoid Arthritis,0.13551503
PheCode:714.1,mainICD + codify,PheCode:714.1,C0039103,Synovitis,0.05149573
PheCode:714.1,mainICD + codify,PheCode:714.1,corrupt_mainNLP,NA,0.04541547
PheCode:714.1,mainICD + codify,PheCode:714.1,RXNORM:214555,etanercept,0.03955847


In [13]:
input.cov.train <- cov_RA_train
input.cov.valid <- cov_RA_valid
out_0 <- KOMAP_corrupt(input.cov.train, input.cov.valid, is.wide = TRUE, target.code, target.cui, 
                nm.disease = 'RA', nm.utl, nm.multi = NULL, nm.corrupt.code = nm.corrupt.code, 
                nm.corrupt.cui = nm.corrupt.cui, dict_RA, pred = FALSE, 
                eval.real = FALSE, eval.sim = FALSE)

Check feature format in `input.cov`...

Num of total feat: 186


Finish estimating coefficients.



In [14]:
out_1 <- KOMAP_corrupt(input.cov.train, input.cov.valid, is.wide = TRUE, target.code, target.cui, 
                       nm.disease = 'RA', nm.utl, nm.multi = NULL, nm.corrupt.code = nm.corrupt.code, 
                       nm.corrupt.cui = nm.corrupt.cui, dict_RA, 
                       codify.feature = codify.feature, nlp.feature = nlp.feature,
                       pred = FALSE, eval.real = FALSE, eval.sim = FALSE)

Check feature format in `input.cov`, `codify.feature` and/or `cuisearch.feature`...

Num of total feat: 186

Num of selected codify feat: 48

Num of selected codify feat after intersection: 37

Num of selected NLP feat: 71

Num of selected NLP feat after intersection: 48


More detailed info...

Num of PheCode in input.cov: 41

Num of PheCode in codify.feature: 19

Num of CCS in input.cov: 4

Num of CCS in codify.feature: 1

Num of LOINC in input.cov: 17

Num of LOINC in codify.feature: 6

Num of RXNORM in input.cov: 40

Num of RXNORM in codify.feature: 18

Num of CUI in input.cov: 81

Num of CUI in cuisearch.feature: 71


Finish estimating coefficients.



In [21]:
out_2 <- KOMAP_corrupt(input.cov.train, input.cov.valid, is.wide = TRUE, target.code, target.cui, 
                       nm.disease = 'RA', nm.utl, nm.multi = NULL, nm.corrupt.code = nm.corrupt.code, 
                       nm.corrupt.cui = nm.corrupt.cui, dict_RA, 
                       codify.feature = codify.feature, nlp.feature = nlp.feature,
                       pred = FALSE, eval.real = FALSE, eval.sim = TRUE,
                       mu0, mu1, var0, var1, prev_Y, B = 10000)
out_2$sim_eval

Check feature format in `input.cov`, `codify.feature` and/or `cuisearch.feature`...

Num of total feat: 186

Num of selected codify feat: 48

Num of selected codify feat after intersection: 37

Num of selected NLP feat: 71

Num of selected NLP feat after intersection: 48


More detailed info...

Num of PheCode in input.cov: 41

Num of PheCode in codify.feature: 19

Num of CCS in input.cov: 4

Num of CCS in codify.feature: 1

Num of LOINC in input.cov: 17

Num of LOINC in codify.feature: 6

Num of RXNORM in input.cov: 40

Num of RXNORM in codify.feature: 18

Num of CUI in input.cov: 81

Num of CUI in cuisearch.feature: 71


Finish estimating coefficients.

Finish estimating AUC.



method,auc
<chr>,<dbl>
mainICD + codify,0.9521138
mainICDNLP + codify & NLP,0.9681361


In [16]:
out_3 <- KOMAP_corrupt(input.cov.train, input.cov.valid, is.wide = TRUE, target.code, target.cui, 
                       nm.disease = 'RA', nm.utl, nm.multi = NULL, nm.corrupt.code = nm.corrupt.code, 
                       nm.corrupt.cui = nm.corrupt.cui, dict_RA, 
                       codify.feature = codify.feature, nlp.feature = nlp.feature,
                       pred = TRUE, eval.real = FALSE, eval.sim = FALSE,
                       dat.part = dat.part, nm.id = nm.id)

Check feature format in `input.cov`, `codify.feature` and/or `cuisearch.feature`...

Num of total feat: 186

Num of selected codify feat: 48

Num of selected codify feat after intersection: 37

Num of selected NLP feat: 71

Num of selected NLP feat after intersection: 48


More detailed info...

Num of PheCode in input.cov: 41

Num of PheCode in codify.feature: 19

Num of CCS in input.cov: 4

Num of CCS in codify.feature: 1

Num of LOINC in input.cov: 17

Num of LOINC in codify.feature: 6

Num of RXNORM in input.cov: 40

Num of RXNORM in codify.feature: 18

Num of CUI in input.cov: 81

Num of CUI in cuisearch.feature: 71


Finish estimating coefficients.

Finish predicting scores.



In [17]:
head(out_3$pred_prob$pred.score)

,patient_num,mainICD + codify,mainICDNLP + codify & NLP
,<chr>,<dbl>,<dbl>
3,s1,0.3395238,-1.1684983
5,s2,-0.2155562,0.5633155
6,s3,-1.4744844,-2.5441391
10,s4,-0.7760274,-1.4015885
13,s5,0.1699729,0.3269727
20,s6,-0.8280096,-0.1911863


In [18]:
head(out_3$pred_prob$pred.prob)

,patient_num,mainICD + codify,mainICDNLP + codify & NLP
,<chr>,<dbl>,<dbl>
1,s1,0.99760586,0.28226168
2,s2,0.74754562,0.99958303
3,s3,0.00993598,0.03222794
4,s4,0.08358144,0.16395525
5,s5,0.98741992,0.99802343
6,s6,0.06658879,0.96145942


In [19]:
head(out_3$pred_prob$pred.cluster)

,patient_num,mainICD + codify,mainICDNLP + codify & NLP
,<chr>,<fct>,<fct>
3,s1,disease,no disease
5,s2,disease,disease
6,s3,no disease,no disease
10,s4,no disease,no disease
13,s5,disease,disease
20,s6,no disease,disease


In [20]:
out_4 <- KOMAP_corrupt(input.cov.train, input.cov.valid, is.wide = TRUE, target.code, target.cui, 
                       nm.disease = 'RA', nm.utl, nm.multi = NULL, nm.corrupt.code = nm.corrupt.code, 
                       nm.corrupt.cui = nm.corrupt.cui, dict_RA, 
                       codify.feature = codify.feature, nlp.feature = nlp.feature,
                       pred = FALSE, eval.real = TRUE, eval.sim = FALSE,
                       dat.part = dat.part, nm.id = nm.id, nm.pi = nm.pi, nm.y = nm.y)
out_4$real_eval

Check feature format in `input.cov`, `codify.feature` and/or `cuisearch.feature`...

Num of total feat: 186

Num of selected codify feat: 48

Num of selected codify feat after intersection: 37

Num of selected NLP feat: 71

Num of selected NLP feat after intersection: 48


More detailed info...

Num of PheCode in input.cov: 41

Num of PheCode in codify.feature: 19

Num of CCS in input.cov: 4

Num of CCS in codify.feature: 1

Num of LOINC in input.cov: 17

Num of LOINC in codify.feature: 6

Num of RXNORM in input.cov: 40

Num of RXNORM in codify.feature: 18

Num of CUI in input.cov: 81

Num of CUI in cuisearch.feature: 71


Finish estimating coefficients.

Finish evaluating model prediction.



method,auc,F_score_max
<chr>,<dbl>,<dbl>
mainICD + codify,0.9364308,0.8669107
mainICDNLP + codify & NLP,0.9581666,0.8904735
